In [1]:
!pip install pyspark

In [5]:
import os
import shutil
from google.colab import files

uploaded = files.upload()

os.makedirs("/content/data_lake/bronze", exist_ok=True)
os.makedirs("/content/data_lake/silver/preprocessed_sales", exist_ok=True)

shutil.copy("/content/sales.csv", "/content/data_lake/bronze/sales.csv")

Saving sales.csv to sales (1).csv


'/content/data_lake/bronze/sales.csv'

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SalesDataLake").getOrCreate()

bronze_path = "/content/data_lake/bronze/sales.csv"

df = spark.read.option("header", True).option("inferSchema", True).csv(bronze_path)

In [7]:
df.printSchema()

df.show(10, truncate=False)

print("Total records:", df.count())

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+--------

In [9]:
from pyspark.sql import functions as F

df.count() - df.dropDuplicates().count()

df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

df.select("category").distinct().show()
df.select("city").distinct().show()

df.filter(F.col("quantity") <= 0).show()
df.filter(F.col("unit_price") <= 0).show()
df.filter((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).show()

df.withColumn(
    "parsed_date",
    F.try_to_timestamp("order_date", F.lit("yyyy-MM-dd"))
).filter(F.col("parsed_date").isNull()).select("order_date").show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

+---------------+
|       category|
+---------------+
|         SPORTS|
|         beauty|
|         Sports|
|        Grocery|
|    Electronics|
|       Clothing|
|          Books|
|    ELECTRONICS|
|Home Appliances|
|      Furniture|
|         sports|
|         Beauty|
|       CLOTHING|
+-----

In [15]:
print("=== 3. Basic Preprocessing - Silver Layer ===")

from pyspark.sql import functions as F

silver_df = df.dropDuplicates()

text_columns = ["customer_id", "customer_name", "product", "category", "payment_method", "city", "state", "order_status"]

for c in text_columns:
    silver_df = silver_df.withColumn(c, F.trim(F.col(c)))

silver_df = silver_df.fillna({"customer_name": "Unknown", "payment_method": "Unknown", "city": "Unknown"})

silver_df = silver_df.withColumn("category", F.initcap("category"))
silver_df = silver_df.withColumn("city", F.initcap("city"))

silver_df = silver_df.withColumn("quantity", F.when(F.col("quantity") <= 0, None).otherwise(F.col("quantity")))

silver_df = silver_df.withColumn("unit_price", F.when(F.col("unit_price") <= 0, None).otherwise(F.col("unit_price")))

silver_df = silver_df.withColumn(
    "discount_percent",
    F.when(
        (F.col("discount_percent") < 0) | (F.col("discount_percent") > 100),
        None
    ).otherwise(F.col("discount_percent"))
)

quantity_median = silver_df.approxQuantile("quantity", [0.5], 0.01)[0]
unit_price_median = silver_df.approxQuantile("unit_price", [0.5], 0.01)[0]
discount_median = silver_df.approxQuantile("discount_percent", [0.5], 0.01)[0]

silver_df = silver_df.fillna({
    "quantity": int(quantity_median),
    "unit_price": float(unit_price_median),
    "discount_percent": float(discount_median)
})

silver_df = silver_df.withColumn(
    "order_date",
    F.to_date(
        F.try_to_timestamp("order_date", F.lit("yyyy-MM-dd"))
    )
)

updated_path = "/content/data_lake/silver/preprocessed_sales"

silver_df.write.mode("overwrite").option("header", True).csv("/content/silver")

=== 3. Basic Preprocessing - Silver Layer ===


In [17]:
print("=== 4. Verification ===")

print("Row count before preprocessing:", df.count())
print("Row count after preprocessing:", silver_df.count())

silver_read = spark.read.parquet(updated_path)

silver_read.show(10, truncate=False)

=== 4. Verification ===
Row count before preprocessing: 1000
Row count after preprocessing: 990
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025